# Job run — AtCoder AWTF 2025 Heuristic — Group Commands & Wall Planning

AtCoder World Tour Finals 2025 Heuristic A. Each organism is a `solve_case(input_text) -> str` candidate, scored on the vendored `seed=0..99` corpus; the objective is to **minimize** the mean absolute score `T + 100 * sum_k Manhattan(final_k, target_k)` (the framework maximizes its negation). The corpus is vendored in-repo — no dataset bootstrap needed.

Two **independent** cluster jobs, each with its own *submit / status / kill* cells. Both set the job `script` to **`scripts/cluster_job.py`** (a real Python file, which is what the cluster's `pytorch2`/torchrun launch expects); it runs the real wrapper on global **rank 0 only** (other ranks exit 0) inside the project env, with all 8 GPUs visible for Ollama:

1. **Normal run** — `cluster_job.py --kind evolve --config-name config_awtf2025_heuristic --env <env> ...` → `run_evolution.sh --seed` (starts Ollama, seeds gen 0, evolves).
2. **Baseline** — `cluster_job.py --kind baseline --config-name baselines/awtf2025_heuristic --env <env> ...` → `run_shinka_baseline.sh` (ShinkaEvolve).

Canonical Comet run-name (set by the repo config, **unchanged**): `awtf2025_heuristic`. Edit `ENV_NAME` / `ENV_MANAGER` in *Common config* to the env you made with `scripts/create_env.sh`. Run *Common config* first, then either job group.

## Common config

In [ ]:
import client_lib

# ---------------------------------------------------------------------------
# Cluster + repo paths
# ---------------------------------------------------------------------------
WORK_DIR = "/home/jovyan/echimbulatov/fork_afedorov/constant_repos/new-optimizer-finding"
BASE_IMAGE = "cr.ai.cloud.ru/2754eb6e-ae19-4123-87ce-06ec3cc96500/job-latentdiffusion:flash-clear"

# ---------------------------------------------------------------------------
# Hardware - a100.{N_GPUS}gpu SKU in the A100-MT region (verbatim from the
# canonical job-run notebook on this cluster). The cluster's pytorch2 job type
# launches `script` under torchrun with one process per GPU; scripts/cluster_job.py
# runs the orchestrator on rank 0 only and keeps all N_GPUS visible so the local
# Ollama instances (gemma 4 31B + qwen 3.5 - see the experiment config) can be
# placed across them.
# ---------------------------------------------------------------------------
N_NODES = 1
N_GPUS = 8
instance_cpu_count = 8
instance_addition = (
    f".{N_GPUS * instance_cpu_count}C.{N_GPUS * 243}G"
    if instance_cpu_count == 8 else ""
)
INSTANCE_TYPE = f"a100.{N_GPUS}gpu{instance_addition}"
REGION = "A100-MT"

_total_gpus = N_NODES * N_GPUS
print(f"Instance: {INSTANCE_TYPE}  (region={REGION})")
print(f"Total GPUs: {_total_gpus}  (= {N_NODES} nodes x {N_GPUS} GPUs)")

# ---------------------------------------------------------------------------
# Project env (create it once with scripts/create_env.sh). The wrapper runs
# inside it via `<ENV_MANAGER> run -n <ENV_NAME>` so lib_runtime.sh resolves
# the right Python. Set ENV_NAME="" to use whatever Python is already on PATH.
# ---------------------------------------------------------------------------
ENV_NAME = "optfind"
ENV_MANAGER = "conda"   # conda | micromamba | mamba

# ---------------------------------------------------------------------------
# Experiment identity. EXP_BASE is the human label used in the job description
# only. The system-visible run identity (the Comet run-name) is owned by the
# repo's Hydra config (cfg.comet.run_name) and left at its canonical default:
#   awtf2025_heuristic
# ---------------------------------------------------------------------------
EXP_BASE = "atcoder-awtf2025"
RUN_CONFIG = "config_awtf2025_heuristic"             # normal evolution-run preset
BASELINE_CONFIG = "baselines/awtf2025_heuristic"   # ShinkaEvolve baseline preset
TASK_ID = ""                   # CO-Bench CO_BENCH_TASK id (UPPER); "" otherwise
NEEDS_COBENCH_BOOTSTRAP = False   # CO-Bench needs the dataset + checkout first

LAUNCHER = f"{WORK_DIR}/scripts/cluster_job.py"

# ---------------------------------------------------------------------------
# Common job env - cluster scaffolding + Comet creds. Same Comet api_key and
# workspace as the canonical notebook (the repo's Hydra config bakes in the
# same api_key too); COMET_PROJECT points at this project's Comet project.
# COMET_RUN_NAME is intentionally NOT set, so the config's canonical per-task
# run-name is preserved.
# ---------------------------------------------------------------------------
common_env = {
    "PROJECT_ROOT": WORK_DIR,
    "PYTHONNOUSERSITE": 1,
    "PIP_USER": "no",
    "NCCL_DEBUG": "INFO",
    "NCCL_IB_TIMEOUT": 23,
    "NCCL_IB_RETRY_CNT": 5,
    "TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC": 3600,
    "MLS_JOB_REGION_NAME": REGION,
    "MLS_JOB_TOTAL_GPU": _total_gpus,
    "CLEARML_CONFIG_FILE": "/home/jovyan/inkoziev/myclearml.conf",
    "COMET_API_KEY": "RrClhd4FveFQKO4qLo4jBjrKu",
    "COMET_WORKSPACE": "dont4rootme",
    "COMET_PROJECT": "new-optimizer-search",
    "COMET_MODE": "online",
    "COMET_LOGGING_CONSOLE": "true",
}

def _env_flags():
    """Shared cluster_job.py flags: env selection + GPU visibility."""
    flags = f" --num-gpus {N_GPUS}"
    if ENV_NAME:
        flags = f" --env {ENV_NAME} --manager {ENV_MANAGER}" + flags
    return flags

def _task_flags():
    """CO-Bench task selector + dataset bootstrap, if applicable."""
    flags = f" --task {TASK_ID}" if TASK_ID else ""
    if NEEDS_COBENCH_BOOTSTRAP:
        flags += " --bootstrap"
    return flags

## Job 1 — Normal evolution run

In [ ]:
# ---------------------------------------------------------------------------
# Normal training run = the organism-first evolution loop, launched on rank 0
# by scripts/cluster_job.py -> scripts/run_evolution.sh:
#   * starts/refreshes the local Ollama instances (scripts/lib_runtime.sh),
#   * with --seed, bootstraps the generation-0 population if missing,
#   * runs seeding + evolution to the configured stop criteria
#     (max_generations / max_organism_creations / per-model token budget).
# `script` is a plain `<file.py> <flags>` line (no cd / bash / && / =), so the
# cluster's torchrun launch runs `python cluster_job.py ...` cleanly.
# ---------------------------------------------------------------------------
run_script = (
    f"{LAUNCHER} --kind evolve --config-name {RUN_CONFIG}"
    f"{_task_flags()}{_env_flags()}"
)
EXP_NAME_RUN = f"{EXP_BASE}-run"
print(f"[{EXP_NAME_RUN}]")
print(run_script)

In [ ]:
run_env = dict(common_env)

# `#ID0137 #rnd` are the user-quota / priority-category tags the cluster
# scheduler needs (verbatim from the canonical job-run notebook); without them
# the job lands in a default bucket with a short wall-time limit.
run_job = client_lib.Job(
    job_desc=f"echimbulatov | {EXP_NAME_RUN} #ID0137 #rnd",
    queue_name="diff",
    base_image=BASE_IMAGE,
    script=run_script,
    n_workers=N_NODES,
    instance_type=INSTANCE_TYPE,
    type="pytorch2",
    preflight_check=True,
    env_variables=run_env,
    region=REGION,
    flags={},
    priority_class="high",
)
run_job.submit()

In [ ]:
while run_job.status() == "Job status=Pending":
    run_job.status()
run_job.logs()

In [ ]:
# run_job.kill()

## Job 2 — ShinkaEvolve baseline

In [ ]:
# ---------------------------------------------------------------------------
# Baseline = ShinkaEvolve over the SAME task evaluator + local Ollama models,
# launched on rank 0 by scripts/cluster_job.py -> scripts/run_shinka_baseline.sh
# (shares the Ollama lifecycle with the run above; invokes src.baselines.shinka.run
# and writes its per-program DB under shinka_runs/).
# ---------------------------------------------------------------------------
baseline_script = (
    f"{LAUNCHER} --kind baseline --config-name {BASELINE_CONFIG}"
    f"{_task_flags()}{_env_flags()}"
)
EXP_NAME_BASELINE = f"{EXP_BASE}-baseline"
print(f"[{EXP_NAME_BASELINE}]")
print(baseline_script)

In [ ]:
baseline_env = dict(common_env)

baseline_job = client_lib.Job(
    job_desc=f"echimbulatov | {EXP_NAME_BASELINE} #ID0137 #rnd",
    queue_name="diff",
    base_image=BASE_IMAGE,
    script=baseline_script,
    n_workers=N_NODES,
    instance_type=INSTANCE_TYPE,
    type="pytorch2",
    preflight_check=True,
    env_variables=baseline_env,
    region=REGION,
    flags={},
    priority_class="high",
)
baseline_job.submit()

In [ ]:
while baseline_job.status() == "Job status=Pending":
    baseline_job.status()
baseline_job.logs()

In [ ]:
# baseline_job.kill()